# 05 — Price Forecasting

Train and evaluate:
- **Prophet** (seasonal decomposition)
- **LSTM** (deep sequence model)
- **Ensemble** (weighted average)

Metrics: MAE · RMSE · MAPE on held-out test set.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from src.data_ingestion.wfp_connector import WFPConnector
from src.preprocessing.data_cleaner import DataCleaner
from src.preprocessing.feature_engineering import FeatureEngineer
from src.models.price_predictor import EnsemblePredictor
from src.utils.helpers import load_config

config = load_config('../config/config.yaml')

raw = WFPConnector(config=config).fetch_price_data()
clean = DataCleaner(config=config).clean(raw)
features = FeatureEngineer(config=config).build_features(clean)
print(f'Features: {features.shape}')

In [ ]:
# ── Fit Ensemble for one product/region ──────────────────────
PRODUCT = 'Tomates'
REGION  = 'Alger'

predictor = EnsemblePredictor(config=config)
predictor.fit(features, PRODUCT, REGION)
print('Ensemble fitted ✅')

In [ ]:
# ── Forecast 12 months ahead ──────────────────────────────────
forecast = predictor.predict(features, PRODUCT, REGION, n_steps=12)
forecast.head()

In [ ]:
# ── Plot Forecast ─────────────────────────────────────────────
hist = features[(features['product'] == PRODUCT) & (features['region'] == REGION)].sort_values('date')

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist['date'], y=hist['price'], name='Historique', line=dict(color='#006233', width=2)))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['ensemble_yhat'], name='Ensemble', line=dict(color='#1565c0', dash='dash', width=2)))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['prophet_yhat'], name='Prophet', line=dict(color='#f57c00', dash='dot')))
fig.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
    y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
    fill='toself', fillcolor='rgba(21,101,192,0.10)',
    line=dict(color='rgba(0,0,0,0)'), name='IC 90%'
))
fig.update_layout(
    title=f'Prévision des Prix — {PRODUCT} / {REGION}',
    xaxis_title='Date', yaxis_title='Prix (DZD)',
    template='plotly_white', height=420, hovermode='x unified'
)
fig.show()

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────
metrics = predictor.evaluate(features, PRODUCT, REGION)
print('\n── Evaluation Metrics ──────────────────────────────')
for model_name, m in metrics.items():
    print(f'{model_name:10s}  MAE={m["MAE"]:7.2f}  RMSE={m["RMSE"]:7.2f}  MAPE={m["MAPE"]:5.1f}%')

In [ ]:
# ── Metrics Bar Chart ─────────────────────────────────────────
metric_df = pd.DataFrame(metrics).T
metric_df[['MAE', 'RMSE']].plot(kind='bar', figsize=(8, 4), rot=0, color=['#1565c0','#d32f2f'], edgecolor='white')
plt.title(f'Comparaison MAE/RMSE — {PRODUCT}/{REGION}')
plt.ylabel('Erreur (DZD)')
plt.tight_layout()
plt.show()

metric_df[['MAPE']].plot(kind='bar', figsize=(6, 3), rot=0, color='#388e3c', edgecolor='white')
plt.title('MAPE (%)')
plt.ylabel('%')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save ensemble model ───────────────────────────────────────
predictor.save('ensemble_v1')
print('Model saved ✅')